# xrVLM - Stage 1 CNN: NIH ChestX-ray14 with ResNet50

This is the ResNet50 replacement for the updated ConvNeXt-Tiny notebook. It keeps the same official NIH split, patient-level validation, weighted multilabel loss, two-phase training, temperature calibration, and validation-tuned thresholds.

Attach the full NIH dataset containing `Data_Entry_2017.csv`, PNG images, `train_val_list.txt`, and `test_list.txt`.

In [ ]:
# Install/import
# If torchvision is already available, the install line is unnecessary.
import os, glob, json, time, random, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image, UnidentifiedImageError
from torchvision import models
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report, hamming_loss, accuracy_score
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Configuration
def find_dataset_root(start='/kaggle/input'):
    for filename in ['Data_Entry_2017.csv', 'Data_Entry_2017_v2020.csv']:
        hits = glob.glob(os.path.join(start, '**', filename), recursive=True)
        if hits: return Path(hits[0]).parent, filename
    return None, None

DATA_ROOT, CSV_FILENAME = find_dataset_root()
if DATA_ROOT is None:
    DATA_ROOT = Path(os.environ.get('NIH_DATA_ROOT', './nih-chest-xrays/data'))
    CSV_FILENAME = 'Data_Entry_2017.csv'
CSV_PATH = DATA_ROOT / CSV_FILENAME
OUTPUT_DIR = Path(os.environ.get('NIH_OUTPUT_DIR', '/kaggle/working/resnet50_nih_official'))
CKPT_DIR, FIGURES_DIR = OUTPUT_DIR / 'checkpoints', OUTPUT_DIR / 'figures'
for d in [OUTPUT_DIR, CKPT_DIR, FIGURES_DIR]: d.mkdir(parents=True, exist_ok=True)

DISEASE_LABELS = ['Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass','Nodule','Pneumonia','Pneumothorax','Consolidation','Edema','Emphysema','Fibrosis','Pleural_Thickening','Hernia']
NUM_CLASSES = len(DISEASE_LABELS)
IMAGE_SIZE = 224
BACKBONE = 'resnet50'
IMAGENET_MEAN, IMAGENET_STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
DROPOUT, TEMPERATURE_INIT = 0.4, 1.5
BATCH_SIZE, NUM_WORKERS = 32, 4
LR_WARMUP, LR_FINETUNE = 1e-3, 2e-5
EPOCHS_WARMUP, EPOCHS_FINETUNE = 4, 6
VAL_PATIENT_FRACTION = 0.10
VERIFY_IMAGES, USE_AMP = False, True
print('Dataset root:', DATA_ROOT); print('CSV:', CSV_PATH)

## 1. Data loading, cleaning, and official NIH split

In [ ]:
assert CSV_PATH.exists(), f'Metadata not found: {CSV_PATH}'
df_raw = pd.read_csv(CSV_PATH)
df = df_raw.drop_duplicates('Image Index').copy()
image_index = {p.name: p for p in DATA_ROOT.rglob('*.png')}
df = df[df['Image Index'].isin(image_index)].copy()
def parse_age(v):
    try: return int(str(v).rstrip('Yy'))
    except ValueError: return np.nan
df['Age'] = df['Patient Age'].map(parse_age)
# Keep all image IDs: official list coverage must not change because of age cleaning.
for label in DISEASE_LABELS: df[label] = df['Finding Labels'].str.contains(label, regex=False).astype(int)
df['No_Finding'] = (df[DISEASE_LABELS].sum(axis=1) == 0).astype(int)

def find_split_file(name):
    hits = glob.glob(os.path.join('/kaggle/input', '**', name), recursive=True)
    if hits: return Path(hits[0])
    p = DATA_ROOT / name
    return p if p.exists() else None
train_val_file, test_file = find_split_file('train_val_list.txt'), find_split_file('test_list.txt')
assert train_val_file and test_file, 'Official list files were not found.'
train_val_ids = {x.strip() for x in train_val_file.read_text().splitlines() if x.strip()}
test_ids = {x.strip() for x in test_file.read_text().splitlines() if x.strip()}
assert not train_val_ids & test_ids
assert train_val_ids | test_ids == set(df['Image Index'])

tv_df = df[df['Image Index'].isin(train_val_ids)].copy()
test_df = df[df['Image Index'].isin(test_ids)].reset_index(drop=True)
rng = np.random.default_rng(SEED)
patients = tv_df['Patient ID'].unique().copy(); rng.shuffle(patients)
n_val = max(1, int(len(patients) * VAL_PATIENT_FRACTION))
val_patients = set(patients[:n_val]); train_patients = set(patients[n_val:])
train_df = tv_df[tv_df['Patient ID'].isin(train_patients)].reset_index(drop=True)
val_df = tv_df[tv_df['Patient ID'].isin(val_patients)].reset_index(drop=True)
assert not set(train_df['Patient ID']) & set(val_df['Patient ID'])
assert not set(tv_df['Patient ID']) & set(test_df['Patient ID'])
print(f'Official lists: train_val={len(train_val_ids):,}, test={len(test_ids):,}')
print(f'Images: train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}')
print(f'Patients: train={train_df["Patient ID"].nunique():,}, val={val_df["Patient ID"].nunique():,}, test={test_df["Patient ID"].nunique():,}')
df.to_csv(OUTPUT_DIR / 'cleaned_metadata.csv', index=False)

In [ ]:
# Dataset characteristics / EDA
counts = df[DISEASE_LABELS + ['No_Finding']].sum().sort_values(ascending=False)
display(counts.to_frame('positive_images'))
plt.figure(figsize=(8,5)); counts[DISEASE_LABELS].sort_values().plot.barh(color='#2a78d6'); plt.title('NIH label frequency'); plt.xlabel('Positive images'); plt.tight_layout(); plt.savefig(FIGURES_DIR / 'label_frequency.png', dpi=140); plt.show()
prevalence = pd.DataFrame({'train %':100*train_df[DISEASE_LABELS].mean(), 'val %':100*val_df[DISEASE_LABELS].mean(), 'test %':100*test_df[DISEASE_LABELS].mean()})
display(prevalence.sort_values('test %', ascending=False))
pos = train_df[DISEASE_LABELS].sum(axis=0).values.astype(np.float32)
neg = len(train_df) - pos
pos_weight_raw = torch.tensor(neg / np.maximum(pos, 1.0), dtype=torch.float32)
pos_weight = torch.sqrt(pos_weight_raw)
print('Using sqrt(pos_weight) to reduce extreme rare-class overprediction.')

## 2. Dataset and transforms

In [ ]:
train_transform = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.RandomRotation(10), T.ColorJitter(brightness=0.15, contrast=0.15), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_transform = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
class ResNet50ChestXrayDataset(Dataset):
    def __init__(self, frame, transform): self.df, self.transform = frame.reset_index(drop=True), transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(image_index[row['Image Index']]).convert('RGB')
        return self.transform(image), torch.tensor(row[DISEASE_LABELS].to_numpy(dtype=np.float32))

train_ds, val_ds, test_ds = [ResNet50ChestXrayDataset(x, t) for x,t in [(train_df,train_transform),(val_df,eval_transform),(test_df,eval_transform)]]
label_sums = train_df[DISEASE_LABELS].sum(axis=1).values
sample_weights = np.where(label_sums > 0, 1.0 / np.maximum(label_sums, 1e-8), 1.0)
sampler = WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.double), len(train_ds), replacement=True)
pin = device.type == 'cuda'
train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=pin, drop_last=True)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
print(next(iter(train_loader))[0].shape)

## 3. ResNet50 model

In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self, init_temp=TEMPERATURE_INIT):
        super().__init__(); self.log_temperature = nn.Parameter(torch.log(torch.tensor(init_temp)))
    @property
    def temperature(self): return self.log_temperature.exp()
    def forward(self, logits): return logits / self.temperature

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=DROPOUT):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(features, num_classes))
        self.temp_scaler = TemperatureScaler()
    def forward_logits(self, x): return self.classifier(self.backbone(x))
    def forward(self, x): return self.temp_scaler(self.forward_logits(x))

model = ResNet50Classifier().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
print(f'ResNet50 parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Two-phase training: classifier warm-up, then full fine-tuning

In [ ]:
def freeze_backbone():
    for name,p in model.named_parameters(): p.requires_grad = ('classifier' in name or 'temp_scaler' in name)
def unfreeze_backbone():
    for p in model.parameters(): p.requires_grad = True
def train_one_epoch(loader, optimizer, scaler):
    model.train(); total = 0.0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', enabled=USE_AMP and device.type=='cuda'):
            loss = criterion(model(images), labels)
        if scaler:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        total += loss.item()
    return total / max(1, len(loader))
@torch.no_grad()
def validate(loader):
    model.eval(); losses=[]; probs=[]; labels=[]
    for images,y in loader:
        images,y=images.to(device),y.to(device)
        logits=model(images).float(); losses.append(criterion(logits,y).item()); probs.append(torch.sigmoid(logits).cpu().numpy()); labels.append(y.cpu().numpy())
    p,y=np.concatenate(probs),np.concatenate(labels)
    auc=[roc_auc_score(y[:,i],p[:,i]) for i in range(NUM_CLASSES) if len(np.unique(y[:,i]))>1]
    return float(np.mean(losses)), float(np.mean(auc)), p, y
def save_checkpoint(path): torch.save({'model_state_dict':model.state_dict(),'classes':DISEASE_LABELS,'backbone':BACKBONE}, path)

scaler = torch.amp.GradScaler('cuda') if (USE_AMP and device.type=='cuda') else None
history=[]; best_val_auroc=-1; best_ckpt_path=CKPT_DIR/'best_model.pth'; epoch_global=0
for phase, epochs, lr in [('warmup',EPOCHS_WARMUP,LR_WARMUP),('finetune',EPOCHS_FINETUNE,LR_FINETUNE)]:
    (freeze_backbone if phase=='warmup' else unfreeze_backbone)()
    optimizer=torch.optim.AdamW(filter(lambda p:p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    for _ in range(epochs):
        epoch_global+=1; t=time.time(); train_loss=train_one_epoch(train_loader,optimizer,scaler); val_loss,val_auc,_,_=validate(val_loader)
        history.append([epoch_global,phase,train_loss,val_loss,val_auc])
        print(f'Epoch {epoch_global:02d} [{phase}] train={train_loss:.4f} val={val_loss:.4f} AUROC={val_auc:.4f} ({time.time()-t:.0f}s)')
        save_checkpoint(CKPT_DIR/'last_model.pth')
        if val_auc>best_val_auroc: best_val_auroc=val_auc; save_checkpoint(best_ckpt_path)
pd.DataFrame(history,columns=['epoch','phase','train_loss','val_loss','val_auroc']).to_csv(OUTPUT_DIR/'training_history.csv',index=False)
print('Best validation AUROC:', best_val_auroc)

## 5. Temperature calibration on validation data

In [ ]:
model.load_state_dict(torch.load(best_ckpt_path, map_location=device)['model_state_dict'])
def tune_temperature(loader):
    model.eval(); [setattr(p,'requires_grad',False) for p in model.parameters()]; model.temp_scaler.log_temperature.requires_grad=True
    logits_all, labels_all = [], []
    with torch.no_grad():
        for x,y in loader: logits_all.append(model.forward_logits(x.to(device))); labels_all.append(y.to(device))
    logits, labels = torch.cat(logits_all), torch.cat(labels_all)
    opt=torch.optim.LBFGS([model.temp_scaler.log_temperature], lr=0.01, max_iter=100); loss_fn=nn.BCEWithLogitsLoss()
    def closure(): opt.zero_grad(); loss=loss_fn(model.temp_scaler(logits),labels); loss.backward(); return loss
    opt.step(closure); [setattr(p,'requires_grad',True) for p in model.parameters()]
    return model.temp_scaler.temperature.item()
final_T=tune_temperature(val_loader)
torch.save({'model_state_dict':model.state_dict(),'temperature':final_T,'classes':DISEASE_LABELS}, CKPT_DIR/'best_model_calibrated.pth')
print(f'Calibrated temperature: {final_T:.4f}')

## 6. Held-out test evaluation and validation-tuned thresholds

In [ ]:
val_loss,val_auc,val_probs,val_labels=validate(val_loader)
test_loss,test_auc,test_probs,test_labels=validate(test_loader)
thresholds=np.full(NUM_CLASSES,0.5)
for i in range(NUM_CLASSES):
    if len(np.unique(val_labels[:,i]))>1:
        candidates=np.arange(0.05,0.95,0.02); scores=[f1_score(val_labels[:,i],val_probs[:,i]>=t,zero_division=0) for t in candidates]; thresholds[i]=candidates[int(np.argmax(scores))]
test_preds=(test_probs>=thresholds[None,:]).astype(int)
rows=[]
for i,label in enumerate(DISEASE_LABELS): rows.append({'class':label,'auroc':roc_auc_score(test_labels[:,i],test_probs[:,i]),'auprc':average_precision_score(test_labels[:,i],test_probs[:,i]),'threshold':thresholds[i]})
metrics=pd.DataFrame(rows); display(metrics); metrics.to_csv(OUTPUT_DIR/'test_metrics.csv',index=False)
print(classification_report(test_labels,test_preds,target_names=DISEASE_LABELS,zero_division=0))
print('Macro AUROC:', test_auc, 'Hamming accuracy:', 1-hamming_loss(test_labels,test_preds), 'Subset accuracy:', accuracy_score(test_labels,test_preds))
np.savez_compressed(OUTPUT_DIR/'predictions.npz',test_probs=test_probs,test_labels=test_labels,val_probs=val_probs,val_labels=val_labels,thresholds=thresholds)
results={'backbone':BACKBONE,'split':'official NIH train_val_list/test_list','seed':SEED,'image_size':IMAGE_SIZE,'best_val_macro_auroc':best_val_auroc,'test_macro_auroc':test_auc,'temperature':final_T,'n_train':len(train_df),'n_val':len(val_df),'n_test':len(test_df)}
with open(OUTPUT_DIR/'results.json','w') as f: json.dump(results,f,indent=2)
print(json.dumps(results,indent=2))